In [ ]:
import json
import numpy as np
import matplotlib.pyplot as plt
from datasets import Dataset
import torch
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
import bitsandbytes as bnb
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification, 
    TrainingArguments, 
    Trainer,
    DataCollatorWithPadding,
    EarlyStoppingCallback
)
import evaluate
from peft import LoraConfig, get_peft_model, TaskType
from collections import Counter
from transformers import BitsAndBytesConfig
from collections import Counter

from dotenv import load_dotenv
import os
load_dotenv()
YOUR_HF_TOKEN = os.getenv("YOUR_HF_TOKEN")

/home/bistreamt/Desktop/master/research 3/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
def combine_text(row):
    # Collect all existing description parts in order
    desc_parts = [row.get(f"case_description_{i}", "") for i in range(1, 9) if row.get(f"case_description_{i}")]
    # Collect all existing justification parts in order
    just_parts = [row.get(f"justification_{i}", "") for i in range(1, 5) if row.get(f"justification_{i}")]
    
    # Prioritize description by putting it first
    return " ".join(desc_parts + just_parts)

In [3]:
class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.get("labels")
        outputs = model(**inputs)
        logits = outputs.get("logits")
            
        loss_fct = torch.nn.CrossEntropyLoss(label_smoothing=0.1)
        loss = loss_fct(logits.view(-1, self.model.config.num_labels), labels.view(-1))
        return (loss, outputs) if return_outputs else loss

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    # predictions = (logits > 0).astype(int)
    return {
        "accuracy": accuracy_score(labels, predictions),
        "f1_macro": f1_score(labels, predictions, average="macro"),
        "precision_macro": precision_score(labels, predictions, average="macro"),
        "recall_macro": recall_score(labels, predictions, average="macro")
    }

In [4]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


In [5]:
with open("data/regional-court-data.json", 'r', encoding='utf-8') as f:
    data = json.load(f)

if isinstance(data, list):
    labels = [item["label"] for item in data if "label" in item]
    
    label_counts = Counter(labels)
    
    print("Number of samples per label:")
    for label, count in sorted(label_counts.items()):
        print(f"{label:12} : {count:4d} samples")
    
    print(f"\nTotal samples in Regional Court Data: {len(data)}")

Number of samples per label:
nevinovat    : 3626 samples
vinovat      : 5244 samples

Total samples in Regional Court Data: 8870


In [6]:
# Define your mapping
path = "data/regional-court-data.json"
label_map = {
    "vinovat": 0,
    "nevinovat": 1
}

def prepare_dataset(json_file_path):
    with open(json_file_path, 'r', encoding='utf-8') as f:
        data = json.load(f)

    formatted_data = []
    for entry in data:
        # Combine description (1-8) and justification (1-4)
        desc = " ".join([entry.get(f"case_description_{i}", "") for i in range(1, 9)]).strip()
        # just = " ".join([entry.get(f"justification_{i}", "") for i in range(1, 5)]).strip()
        
        formatted_data.append({
            "text": f"{desc}", # Description prioritized by order
            "label": label_map[entry["label"]]
        })
    
    # Split into train and validation (85/15)
    train_data, val_data = train_test_split(
        formatted_data, 
        test_size=0.15, 
        stratify=[d["label"] for d in formatted_data], # Keep class ratios same
        random_state=42
    )
    
    return Dataset.from_list(train_data), Dataset.from_list(val_data)

train_raw, val_raw = prepare_dataset(path)

# RoBERTBase

In [7]:
model_name = "dumitrescustefan/bert-base-romanian-cased-v1"

tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize_function(examples):
    # tokenizer.pad_token = tokenizer.eos_token
    # tokenizer.padding_side = "right"
    return tokenizer(examples["text"], truncation=True, padding="max_length", max_length=512)

tokenized_train = train_raw.map(tokenize_function, batched=True)
tokenized_val = val_raw.map(tokenize_function, batched=True)


Map: 100%|██████████| 1331/1331 [00:01<00:00, 1170.41 examples/s]


In [9]:
# !pip install textattack

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 6.6 MB/s  0:00:00 eta 0:00:01
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.2/19.2 MB 8.7 MB/s  0:00:02 eta 0:00:01
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 981.5/981.5 kB 4.9 MB/s  0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to bui

In [11]:
model = AutoModelForSequenceClassification.from_pretrained(
    model_name, 
    num_labels=2
    ).to(device) 
print(f"Using device: {device}")

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at dumitrescustefan/bert-base-romanian-cased-v1 and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Using device: cuda


In [13]:
training_args = TrainingArguments(
    output_dir="results/good/district-ro-bert-base",
    learning_rate=2e-5,
    lr_scheduler_type="cosine",
    warmup_steps=200,
    warmup_ratio=0.1,
    per_device_train_batch_size=8,
    gradient_accumulation_steps=2,
    num_train_epochs=5,
    weight_decay=0.1,
    fp16=True,
    logging_steps=50,
    eval_steps=50,
    eval_strategy="steps",
    save_strategy="epoch",
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    compute_metrics=compute_metrics,
    data_collator=DataCollatorWithPadding(tokenizer=tokenizer)
)

trainer.train()

Step,Training Loss,Validation Loss,Accuracy,F1 Macro,Precision Macro,Recall Macro
50,0.702800,0.653292,0.594290,0.392583,0.597101,0.505947
100,0.652800,0.612365,0.652893,0.571184,0.671737,0.592963
150,0.620700,0.668314,0.630353,0.500311,0.680673,0.555740
200,0.577000,0.549262,0.721262,0.683078,0.736457,0.680576
250,0.528200,0.518348,0.746056,0.739761,0.738260,0.742408
300,0.530200,0.523323,0.749812,0.742111,0.741345,0.743030
350,0.475100,0.494338,0.766341,0.764160,0.765475,0.774602
400,0.502400,0.472626,0.779113,0.768809,0.772912,0.766105
450,0.510200,0.561550,0.722014,0.662995,0.783817,0.667873
500,0.488900,0.451289,0.784373,0.771835,0.781291,0.767147


TrainOutput(global_step=2360, training_loss=0.34182550932391215, metrics={'train_runtime': 1411.2783, 'train_samples_per_second': 26.71, 'train_steps_per_second': 1.672, 'total_flos': 9917971231795200.0, 'train_loss': 0.34182550932391215, 'epoch': 5.0})

In [14]:
from transformers import AutoModelForSequenceClassification

# Load the trained model from checkpoint

checkpoint_path = "results/good/district-ro-bert-base/checkpoint-1888"
model = AutoModelForSequenceClassification.from_pretrained(checkpoint_path).to(device)
print(f"Model loaded from {checkpoint_path}")

Model loaded from results/good/district-ro-bert-base/checkpoint-1888


In [31]:
import torch
from textattack.models.wrappers import HuggingFaceModelWrapper
from textattack.datasets import Dataset
from textattack import Attack, AttackArgs, Attacker
from textattack.search_methods import GreedyWordSwapWIR
from textattack.transformations import WordSwapNeighboringCharacterSwap
from textattack.constraints.pre_transformation import RepeatModification, StopwordModification
from textattack.goal_functions import UntargetedClassification  # <--- Added this
from textattack import Attacker

# 1. Force evaluation settings and batch sizing
model.eval()
model_wrapper = HuggingFaceModelWrapper(model, tokenizer) 

# 2. Set up the Goal Function (Untargeted means: change the prediction to anything else)
goal_function = UntargetedClassification(model_wrapper)

print("Goal function set up successfully.")

# 3. Use a Character-Swap Transformation (No downloads required!)
transformation = WordSwapNeighboringCharacterSwap(
    random_one=True, 
    skip_first_char=True, 
    skip_last_char=True
)

print("Transformation set up successfully.")

# Stopwords to avoid breaking fundamental legal tokens (e.g., "Art.", "Cod")
constraints = [
    RepeatModification(),
    StopwordModification()
]

print("Constraints set up successfully.")

# 4. Initialize Search Method with the Goal Function
search_method = GreedyWordSwapWIR(wir_method="unk")

print("Search method set up successfully.")

# 5. Construct the custom attack passing all components correctly
attack = Attack(goal_function, constraints, transformation, search_method)

print("Attack constructed successfully.")



textattack: Unknown if model of class <class 'transformers.models.bert.modeling_bert.BertForSequenceClassification'> compatible with goal function <class 'textattack.goal_functions.classification.untargeted_classification.UntargetedClassification'>.


Goal function set up successfully.
Transformation set up successfully.
Constraints set up successfully.
Search method set up successfully.
Attack constructed successfully.


In [32]:
# 6. Format dataset
attack_samples = []
for item in val_raw.select(range(200)):
    truncated_text = " ".join(item["text"].split()[:512]) 
    attack_samples.append((truncated_text, int(item["label"])))
textattack_dataset = Dataset(attack_samples)

# 7. Force execution over a wider evaluation block to confirm the ~20% rate
attack_args = AttackArgs(
    num_examples=50, # Evaluates until it gets 30 valid samples to build a stable metric
    attack_n=True
)

attacker = Attacker(attack, dataset=textattack_dataset, attack_args=attack_args)
attacker.attack_dataset()

Attack(
  (search_method): GreedyWordSwapWIR(
    (wir_method):  unk
  )
  (goal_function):  UntargetedClassification
  (transformation):  WordSwapNeighboringCharacterSwap(
    (random_one):  True
  )
  (constraints): 
    (0): RepeatModification
    (1): StopwordModification
  (is_black_box):  True
) 



--------------------------------------------- Result 1 ---------------------------------------------
[[1 (85%)]] --> [[[FAILED]]]

xxxxNUMExxxx, pretinzînd că este avocata xxxxNUMExxxx, care activează în baza licenței nr. 2295 din 21.12.2011, urmărind scop de profit, prin înșelăciune și abuz de încredere, a încheiat contract de asistență juridică, nr. 26 din 16.01.2014, cu xxxxNUMExxxx, succesorul părții vătămate pe cauza penală nr. 2013320554, de învinuire a lui Petcu Viorel, în comiterea infracțiunii prevăzute de art. 145 alin. (1) Cod Penal, și a prezentat în instanța de judecată, mandatul nr. 0474957 din 16.01.2014, prin care s­a împuternicit să acorde lui xxxxNUMExxxx asistență juridică în toate instanțele de judecată, dobîndind ilicit de la ultimul mijloace bănești în sumă de 5000 lei. În cadrul părţii pregătitoare a şedinţei de judecată, partea vătămată xxxxNUMExxxx a depus cerere prin care a solicitat încetarea procesului penal din motivul că s­a împăcat cu inculpata xxxxNUMExx

--------------------------------------------- Result 2 ---------------------------------------------
[[1 (98%)]] --> [[0 (50%)]]

[[Prin]] sentinţa nominalizată xxxxNUMExxxx a fost [[condamnat]] [[pentru]], că în noaptea de 08.10.2012, intenționat, cu scopul sustragerii bunurilor altei persoane, aflîndu­se în casa lui G R de pe str.r.din or.[[xxxxORAS_SATxxxx]], a atacat­o pe G R cu un cuțit de bucătărie și i­a aplicat acesteia mai multe lovituri în regiunea [[toracelui]] și abdomenului cauzîndu­i astfel leziuni corporale grave periculoase pentru viață sub formă de plagă [[nepenetrantă]] la peretele abdominal anterior pe stînga cu dimensiunile 2,5 cm x 0,8 cm și [[adîncimea]] de 2,3 cm, trei plăgi [[nepetetrante]] în regiunea [[toracelui]] pe stînga la nivelul intercostal 3­5 linia anatomică [[parasternală]] cu dimensiunile 1,6 cm x 0,5 cm, 1,8 cm x0,5 cm, 2,9 cm x 0,7 cm și adfîncimea de pînă la 1,8 cm, o plagă penetrabtă în regiunea toracelui pe stînga la nivelul intercostal 3­5 lini

--------------------------------------------- Result 3 ---------------------------------------------
[[0 (100%)]] --> [[[FAILED]]]

Prin sentinţa Judecătoriei Soroca din 21.10.2004 x a fost condamnat în baza art.172 al.(3) pct.c) Cod penal RM la pedeapsă sub formă de 15 ani închisoare în peniteciar de tip închis. Începutul termenului –03.09.2004, sfîrşitul termenului ­02.09.2019. La data de 28.06.2016 șeful­adjunct al Penitenciarului nr.6 Soroca Cebotari Alexandru a înaintat în instanța de judecată demers, prin care a solicitat examinarea posibilității liberării condiționate de pedeapsă înainte de termen a condamnatului x. În motivarea demersului a indicat, că condamnatul x a ispășit ¾ din pedeapsa stabilită în conformitate cu legislația în vigoare, se caraterizează satisfăcător, pe perioada executării pedepsei a fost stimulat de două ori, a fost sancționat disciplinar o singură dată cu mustrare la data de 21.12.2005, sancțiune care la moment este stinsă, anterior a participat la munci

--------------------------------------------- Result 4 ---------------------------------------------
[[1 (84%)]] --> [[0 (74%)]]

Inculpații xxxxNUMExxxx și xxxxNUMExxxx se învinuiesc de că către organul de urmărire penală de faptul că, la 10.05.2014, ora 15:30, fiind efectuată de către colaboratorii Penitenciarului nr. 17 Rezina, percheziţia corporală a deţinutului xxxxNUMExxxx, la finele întrevederii de scurtă durată a acestuia cu soţia sa xxxxNUMExxxx, s­a depistat la acesta un pachet din material polimeric de culoare galben­transparent, o seringă medicală de unică folosinţă capacitatea de 20 ml., două „bile” din material polimeric transparent, în care în care se afla o masă vegetală uscată, de culoare verde, mărunţită, cu miros specific, asemănător cu cel de cînepă. Conform concluziei raportului de constatare tehnico­ştiinţifică nr. 4444 din 20.05.2014, masele vegetale de culoare verde, mărunţită, uscată, prezentată la examinare într­un pachet din material polimeric de culoare galb

...

--------------------------------------------- Result 57 ---------------------------------------------
[[0 (100%)]] --> [[1 (51%)]]

xxxxNUMExxxx la 24.12.[[2014]], aproximativ la ora 23.30 minute, conducînd autoturismul de model “Ford [[Escort]]”, cu numărul de înmatriculare BLCE 271, în a s. Drăgăneşti r­l Sîngerei, a fost stopat de către colaborotorii de poliţie şi fiind suspectat în consum de băuturi alcoolice, a fost supus testării alcoolscopice cu ajutorul tehnic “Drager­[[Alcotest]] 6810”, fiind stabilită concentraţia vaporilor de alcool în aereul expirat de 0,52 mg/l, ceea ce conform art. 13412 alin. (3) din Codul penal, se stare de ebrietate alcoolică cu [[grad]] avansat. Fiind audiat în ședința de judecată, inculpatul xxxxNUMExxxx a recunoscut vina integral, [[declarînd]] că, la 24.12.2014, aproximativ pe la o conducea automobilul său de model „Ford [[Escort]]” cu numerele de înregistrare BL CE 271 din s. Serbeşti r. Floreşti de la nişte prieteni spre s. Drăgăneşti r­l Sîngere

--------------------------------------------- Result 58 ---------------------------------------------
[[0 (100%)]] --> [[[FAILED]]]

Inculpatul xxxxNUMExxxx Nichita, la data de 30.10.2013, pe la orele 16.00, în rîpa din preajma s. Lalova rl. Rezina, în timpul unui conflict spantan apărut pe fondalul relaţiilor ostile persistante i­a aplicat cet. Josan Victor, multiple lovituri cu pumnii în regiunea feţii, în rezultatul cărora lui Josan Igor i­au fost provocate, potrivit raportului de expertiză medico­ legală nr. 397 D din 27.11.2013, vătămări corporale MEDII sub formă de fractură a mandibulei în regiunea unghiului pe dreapt, comoţie cerebrală, ce duc la o dereglare a sănătăţii de lungă durată. Inculpatul xxxxNUMExxxx Nichita pînă la începerea cercetării judecătoreşti a declarat personal prin înscris autentic, că recunoaşte săvîrşirea faptelor indicate în rechizitoriu şi solicită ca judecata să se facă pe baza probelor administarte în faza de urmărîre penală pe care le cunoaşte şi asupr

--------------------------------------------- Result 59 ---------------------------------------------
[[1 (86%)]] --> [[[FAILED]]]

Prin sentința nominalizată xxxxNUMExxxx a fost recunoscută culpabilă și condamnataă pentru, că în luna decembrie 2008 prin înțelegere prealabilă și împreună cu Iftodi Olesea, încurajînd­o și convingînd­o pe xxxNUMExxxcă va obține venituri considerabile pe cont propriu ca rezultat al practicării prostituției în Turcia, trezindu­i interesul față de această îndeletnicire, îndemnînd­o și încurajînd­ o, au determinat­o să­și dea consimțămîntul de a practica acest gen de activitate. Respectivele acțiuni a inculpatei fiind încadrate de către instanța de fond la art. 220 alin. (1) Cod penal cu calificativele ”îndemnul și determinarea la prostituție, înlesnirea practicii prostituției, tragerea de foloase de pe urma practricării prostituției” (f.d. 246­247). Totodată, prin învinuirea formulatră în rechizitoriu, xxxxNUMExxxx a fost acuzată de faptul, că: ”indentificî

[Succeeded / Failed / Skipped / Total] 24 / 26 / 10 / 60: 100%|██████████| 50/50 [09:58<00:00, 11.98s/it]

--------------------------------------------- Result 60 ---------------------------------------------
[[0 (99%)]] --> [[1 (63%)]]

[[Inculpatul]] xxxxNUMExxxx activînd în calitate de inspector­şofer (grupa operativă) al Dispetceratului pază al Filialei [[Rezina]] Întreprinderii de Stat „Servicii Pază Republicii Moldova , în perioada de timp începînd cu luna decembrie 2010, susţinînd că are influenţă asupra unor funcţionari din cadrul Comisariatului raional de poliţie [[Rezin]] Procuratura raionului Rezina , în gestiunea şi conducerea cărora se afla cauza penală nr.[[20100297]] pornită în privinţa cet.xxxxCUSTOM2xxxx Mihai, Muntean Vasi [[xxxxCUSTOM1xxxx]] Diana şi xxxxCUSTOM1xxxx Alexei conform art.217/1 al.(4) lit.d) CP, în scopul de ai [[detrmina]] pe aceştea la îndeplinirtea unor acţiuni ce într obligaţiunile de serviciu legate de încetarea urmărirei penale pe cauza penală dată , prin extorcare a pretins bani în sumă de 800 dolari SUA de la cet.xxxxCUSTOM1x Alexei , care ulterior la

In [33]:
# 6. Format dataset
attack_samples = []
for item in val_raw.select(range(200)):
    truncated_text = " ".join(item["text"].split()[:512]) 
    attack_samples.append((truncated_text, int(item["label"])))
textattack_dataset = Dataset(attack_samples)

# 7. Force execution over a wider evaluation block to confirm the ~20% rate
attack_args = AttackArgs(
    num_examples=100, # Evaluates until it gets 30 valid samples to build a stable metric
    attack_n=True
)

attacker = Attacker(attack, dataset=textattack_dataset, attack_args=attack_args)
attacker.attack_dataset()

Attack(
  (search_method): GreedyWordSwapWIR(
    (wir_method):  unk
  )
  (goal_function):  UntargetedClassification
  (transformation):  WordSwapNeighboringCharacterSwap(
    (random_one):  True
  )
  (constraints): 
    (0): RepeatModification
    (1): StopwordModification
  (is_black_box):  True
) 



[Succeeded / Failed / Skipped / Total] 0 / 1 / 0 / 1:   1%|          | 1/100 [00:13<22:58, 13.92s/it]

--------------------------------------------- Result 1 ---------------------------------------------
[[1 (85%)]] --> [[[FAILED]]]

xxxxNUMExxxx, pretinzînd că este avocata xxxxNUMExxxx, care activează în baza licenței nr. 2295 din 21.12.2011, urmărind scop de profit, prin înșelăciune și abuz de încredere, a încheiat contract de asistență juridică, nr. 26 din 16.01.2014, cu xxxxNUMExxxx, succesorul părții vătămate pe cauza penală nr. 2013320554, de învinuire a lui Petcu Viorel, în comiterea infracțiunii prevăzute de art. 145 alin. (1) Cod Penal, și a prezentat în instanța de judecată, mandatul nr. 0474957 din 16.01.2014, prin care s­a împuternicit să acorde lui xxxxNUMExxxx asistență juridică în toate instanțele de judecată, dobîndind ilicit de la ultimul mijloace bănești în sumă de 5000 lei. În cadrul părţii pregătitoare a şedinţei de judecată, partea vătămată xxxxNUMExxxx a depus cerere prin care a solicitat încetarea procesului penal din motivul că s­a împăcat cu inculpata xxxxNUMExx

[Succeeded / Failed / Skipped / Total] 1 / 1 / 0 / 2:   2%|▏         | 2/100 [00:23<19:27, 11.92s/it]

--------------------------------------------- Result 2 ---------------------------------------------
[[1 (98%)]] --> [[0 (50%)]]

[[Prin]] sentinţa nominalizată xxxxNUMExxxx a fost [[condamnat]] [[pentru]], că în noaptea de 08.10.2012, intenționat, cu scopul sustragerii bunurilor altei persoane, aflîndu­se în casa lui G R de pe str.r.din or.[[xxxxORAS_SATxxxx]], a atacat­o pe G R cu un cuțit de bucătărie și i­a aplicat acesteia mai multe lovituri în regiunea [[toracelui]] și abdomenului cauzîndu­i astfel leziuni corporale grave periculoase pentru viață sub formă de plagă [[nepenetrantă]] la peretele abdominal anterior pe stînga cu dimensiunile 2,5 cm x 0,8 cm și [[adîncimea]] de 2,3 cm, trei plăgi [[nepetetrante]] în regiunea [[toracelui]] pe stînga la nivelul intercostal 3­5 linia anatomică [[parasternală]] cu dimensiunile 1,6 cm x 0,5 cm, 1,8 cm x0,5 cm, 2,9 cm x 0,7 cm și adfîncimea de pînă la 1,8 cm, o plagă penetrabtă în regiunea toracelui pe stînga la nivelul intercostal 3­5 lini

...

--------------------------------------------- Result 129 ---------------------------------------------
[[1 (96%)]] --> [[0 (78%)]]

Pentru a se expune în sensul celor expuse, instanţa de fond a reţinut că xxxxNUMExxxx, la 02.11.2014, aproximativ pe la orele 22.oo, aflîndu­se în stare de ebrietate produsă de alcool, urmărind intenția de a aplica violență psihică și fizică în privința unui membru a familiei, în propria casă situată în satul xxxxORAS_SATxxxx, raionul , sub pretextul unor conflicte cu soția sa xxxxNUMExxxx, intenționat, în scop de atingere a vieții, integrității corporale și psihologice a unui membru al familiei, în timpul unei cerți, întru demonstrarea supremației asupra soției, a manifestat violență psihică în privința acesteia, a agresat­o verbal și a intimidat­o, numind­o cu cuvinte necenzurate și injurioase, provocîndu­i sentimentul de temere, după care a aplicat violență fizică în privința ei, exprimată prin aplicarea unei lovituri de la spate cu o vargă metalică pes

[Succeeded / Failed / Skipped / Total] 47 / 52 / 31 / 130:  99%|█████████▉| 99/100 [18:48<00:11, 11.39s/it]

--------------------------------------------- Result 130 ---------------------------------------------
[[1 (88%)]] --> [[0 (51%)]]

Prin sentința Judecătoriei Drochia sediul [[Central]] din 27.12.2018, Bejan Ion ***** a [[fost]] recunoscut vinovat de comiterea infracțiunii prevăzute de art.186 alin.(2) lit.b), c), d) Codul Penal RM și ca măsură de pedeapsă, i-a fost aplicată 200 (două sute) ore muncă neremunerată în folosul cominității. Maiorov Serghei ***** a fost recunoscut vinovat de comiterea infracțiunii prevăzute de art.186 alin.(2) lit.b), c), d) Codul Penal RM și ca măsură de pedeapsă, i-a fost aplicată 200 (două sute) ore muncă neremunerată în folosul cominității. Tintiuc Andrei ***** a fost recunoscut [[vinovat]] de comiterea infracțiunilor prevăzute de art.186 alin.(2) lit.b), c), d), art.217 alin.(2), art.1921 alin.(2) lit.a), art.187 alin.(2) lit.d), e), f) Cod Penal al RM și ca măsură de pedeapsă, i-a fost aplicaă : - în baza art.186 alin.(2) lit.b), c), d) Codul Penal al

[Succeeded / Failed / Skipped / Total] 48 / 52 / 31 / 131: 100%|██████████| 100/100 [18:57<00:00, 11.37s/it]

--------------------------------------------- Result 131 ---------------------------------------------
[[0 (93%)]] --> [[1 (58%)]]

Pentru a se pronunţa în sensul celor expuse instanţa de fond a reţinut că inculpatul Cozarevici Semion Mihail la o dată neidentificată de organul de urmărire penală, în perioada de timp cuprinsă între 23.03.2017 până la 15.04.2017, aflându-se pe terenul agricol GŢ „Dutca Vădim”, [[care-i]] aparţine cu drept de proprietate lui Dutca Vadim, amplasat în extravilanul satului Grigorăuca r-nul Sîngerei, acţionând intenţionat, cu scopul sustragerii bunurilor altei persoane, înţelegând caracterul prejudiciabil al acţiunilor sale, pe ascuns a sustras 21 pomi de nuci altoiţi în valoare de 9 Euro fiecare, cauzând părţii vătămate o pagubă materială considerabilă în sumă totală de 189 Euro, echivalentul a 3931 lei MD. Sentinţa instanţei de fond a [[fost]] atacată cu apel în termen de către [[inculpatul]] Cozarevici Semion, care în motivare invocă că nu este deacord cu 

# RoGPT2

In [8]:
def prepare_dataset(path):
    with open(path, 'r', encoding='utf-8') as f:
        data = json.load(f)

    label_map = {
        "vinovat": 0,
        "nevinovat": 1
    }

    formatted_data = []
    for entry in data:
        desc = " ".join([entry.get(f"case_description_{i}", "") for i in range(1, 9)]).strip()

        # --- PROMPT ENGINEERING START ---
        instructional_prompt = (
            f"Ești un expert legal și task-ul tău este de a analiza "
            f"un caz juridic și de a decide verdictul corect. "
            f"Analizează pas cu pas informațiile oferite înainte "
            f"de a formula răspunsul final. "
            f"Decide verdictul acestui caz juridic:\n"
            f"Ai de ales între două răspunsuri: admis sau respins.\n"
            f"Descriere caz: {desc}\n" 
        )

        # --- PROMPT ENGINEERING END ---
        
        formatted_data.append({
            "text": instructional_prompt, 
            "label": label_map[entry["label"]]
        })

    train_data, val_data = train_test_split(
        formatted_data, 
        test_size=0.15, 
        stratify=[d["label"] for d in formatted_data], # Keep class ratios same
        random_state=42
    )
    
    return Dataset.from_list(train_data), Dataset.from_list(val_data)

train_raw, val_raw = prepare_dataset("data/regional-court-data.json")

In [9]:
model_name = "readerbench/RoGPT2-base"

tokenizer = AutoTokenizer.from_pretrained(model_name)

tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

def tokenize_function(examples):
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = "right"
    return tokenizer(examples["text"], truncation=True, padding="max_length", max_length=512)

tokenized_train = train_raw.map(tokenize_function, batched=True)
tokenized_val = val_raw.map(tokenize_function, batched=True)

model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=2
)

# Important for GPT2 padding
model.config.pad_token_id = tokenizer.pad_token_id

model = model.to(device)

print(f"Using device: {device}")

training_args = TrainingArguments(
    output_dir="results/good/dist-ro-gpt2-base",
    learning_rate=2e-5,
    lr_scheduler_type="cosine",
    warmup_steps=200,
    warmup_ratio=0.1,
    per_device_train_batch_size=8,
    gradient_accumulation_steps=2,
    num_train_epochs=5,
    weight_decay=0.1,
    fp16=True,
    logging_steps=50,
    eval_steps=50,
    eval_strategy="steps",
    save_strategy="epoch",
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    compute_metrics=compute_metrics,
    data_collator=DataCollatorWithPadding(tokenizer=tokenizer)
)

trainer.train()

Map: 100%|██████████| 1331/1331 [00:01<00:00, 1056.42 examples/s]
Some weights of GPT2ForSequenceClassification were not initialized from the model checkpoint at readerbench/RoGPT2-base and are newly initialized: ['score.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Using device: cuda


Step,Training Loss,Validation Loss,Accuracy,F1 Macro,Precision Macro,Recall Macro
50,0.738900,0.716965,0.535687,0.500128,0.504173,0.503785
100,0.717300,0.685746,0.569497,0.515593,0.533013,0.526131
150,0.680500,0.658291,0.603306,0.554387,0.577297,0.561532
200,0.649000,0.630537,0.650639,0.609075,0.638260,0.611490
250,0.603700,0.613136,0.667168,0.658504,0.657670,0.660090
300,0.586500,0.568725,0.720511,0.714380,0.713028,0.717685
350,0.537300,0.542813,0.743050,0.729201,0.735566,0.725960
400,0.514100,0.531319,0.751315,0.742667,0.742736,0.742598
450,0.534400,0.573246,0.740045,0.701738,0.767660,0.697878
500,0.486400,0.508888,0.761833,0.744997,0.760147,0.739573


KeyboardInterrupt: 

In [10]:
checkpoint_path = "results/good/dist-ro-gpt2-base/checkpoint-1416"
model = AutoModelForSequenceClassification.from_pretrained(checkpoint_path).to(device)
model.eval()
print(f"Model loaded from {checkpoint_path}")

Model loaded from results/good/dist-ro-gpt2-base/checkpoint-1416


In [11]:
import pandas as pd
from transformers import Trainer

# 1. Define Jamming Strings
# Strategy A: Heavy boilerplate text to crowd out the sequence length limit
PREFIX_JAMMING_TEXT = "Incheiere text administrativ. Monitorul Oficial nr. 1. Procedura legala in vigoare. Document intern de arhiva judiciara. " * 8 

# Strategy B: A disruptive token to inject inside the text body
INTERLEAVED_JAMMING_TOKEN = " [NOTIFICARE-SISTEM] "

# 2. Attack Functions
def apply_prefix_jamming(raw_dataset):
    def jam_prefix(example):
        example["text"] = PREFIX_JAMMING_TEXT + example["text"]
        return example
    return raw_dataset.map(jam_prefix).map(tokenize_function, batched=True)

def apply_interleaved_jamming(raw_dataset):
    def jam_interleave(example):
        words = example["text"].split()
        # Inject the jamming token every 5 words
        jammed_words = []
        for i, word in enumerate(words):
            jammed_words.append(word)
            if (i + 1) % 5 == 0:
                jammed_words.append(INTERLEAVED_JAMMING_TOKEN)
        example["text"] = " ".join(jammed_words)
        return example
    return raw_dataset.map(jam_interleave).map(tokenize_function, batched=True)

# 3. Generate Jammed Validation Sets
tokenized_val_prefix_jam = apply_prefix_jamming(val_raw)
tokenized_val_interleaved_jam = apply_interleaved_jamming(val_raw)

# 4. Evaluate using your existing RoGPT2 Trainer
print("Evaluating Baseline (Clean)...")
clean_metrics = trainer.evaluate(eval_dataset=tokenized_val)

print("Evaluating Prefix Jamming (Context Eviction)...")
prefix_jam_metrics = trainer.evaluate(eval_dataset=tokenized_val_prefix_jam)

print("Evaluating Interleaved Jamming (Attention Dilution)...")
interleaved_jam_metrics = trainer.evaluate(eval_dataset=tokenized_val_interleaved_jam)

# 5. Compile into a Thesis-ready DataFrame
results_df = pd.DataFrame({
    "Metric": clean_metrics.keys(),
    "Clean Baseline": clean_metrics.values(),
    "Prefix Jamming": prefix_jam_metrics.values(),
    "Interleaved Jamming": interleaved_jam_metrics.values()
})
print(results_df)

Map: 100%|██████████| 1331/1331 [00:02<00:00, 506.82 examples/s]

Evaluating Baseline (Clean)...


Evaluating Prefix Jamming (Context Eviction)...
Evaluating Interleaved Jamming (Attention Dilution)...
                 Metric  Clean Baseline  Prefix Jamming  Interleaved Jamming
0             eval_loss        0.704502        0.790657             0.776139
1         eval_accuracy        0.790383        0.731781             0.673929
2         eval_f1_macro        0.787231        0.729277             0.629041
3  eval_precision_macro        0.785887        0.730911             0.672513
4     eval_recall_macro        0.794649        0.738850             0.631469


# RAG Attacks

In [12]:
import json

label_map = {
    "vinovat": 0,
    "nevinovat": 1
}

def prepare_dataset(json_file_path):
    with open(json_file_path, 'r', encoding='utf-8') as f:
        data = json.load(f)
    

    formatted_data = []
    for entry in data:
        entry.pop("source_path", None)
        # Combine description (1-8) and justification (1-4)
        desc = " ".join([entry.get(f"case_description_{i}", "") for i in range(1, 9)]).strip()
        just = " ".join(entry.get(f"sentence", "")).strip()
        sentence = " ".join(entry.get(f"sentence", "")).strip()
        if len(desc) < 5610:
            formatted_data.append({
                "text": f"{desc}", # Description prioritized by order
                "just": f"{just}", # Justification
                "sentence": f"{sentence}", # Sentence prioritized by order
                "label": label_map[entry["label"]]
            })
    
    # Split into train and validation (85/15)
    train_data, val_data = train_test_split(
        formatted_data, 
        test_size=0.0499, 
        stratify=[d["label"] for d in formatted_data], # Keep class ratios same
        random_state=42
    )
    
    return Dataset.from_list(train_data), Dataset.from_list(val_data)

rag_dataset, val_dataset = prepare_dataset("data/regional-court-data.json")

In [13]:
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer
from datasets import Dataset

# 1. Setup Model (Optimized for Romanian nuances)
# This model is excellent for Romanian and 50+ other languages.
model_name = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
embedder = SentenceTransformer(model_name)

def build_rag_system(rag_dataset):
    # 2. Encode the Knowledge Base
    print("Encoding knowledge base...")
    kb_texts = rag_dataset['text']
    kb_embeddings = embedder.encode(kb_texts, convert_to_numpy=True, show_progress_bar=True)
    
    # 3. Initialize FAISS Index
    dimension = kb_embeddings.shape[1]
    index = faiss.IndexFlatL2(dimension)
    index.add(kb_embeddings)
    
    return index, kb_texts, rag_dataset['label']

# Initialize the system
faiss_index, kb_texts, kb_labels = build_rag_system(rag_dataset)

def run_inference(query_text, k):
    # 4. Retrieval
    query_embedding = embedder.encode([query_text], convert_to_numpy=True)
    distances, indices = faiss_index.search(query_embedding, k)
    
    # Get retrieved neighbors
    retrieved_labels = [kb_labels[i] for i in indices[0]]
    retrieved_texts = [kb_texts[i] for i in indices[0]]
    
    # 5. Logic for Label, Confidence, and Explanation
    # We calculate confidence based on label agreement and distance
    label_0_count = retrieved_labels.count(0)
    label_1_count = retrieved_labels.count(1)
    
    predicted_label = 1 if label_1_count >= label_0_count else 0
    
    # Simple confidence score based on retrieval consensus
    consensus = max(label_0_count, label_1_count) / k
    # Soften score based on distance (closer = more confident)
    avg_distance = np.mean(distances[0])
    confidence_score = round(consensus * (1 / (1 + avg_distance * 0.1)), 2)
    
    # Construct Romanian explanation
    explanation = f"Bazat pe cele mai similare {k} exemple găsite: "
    explanation += f"{label_1_count} indică eticheta 1 și {label_0_count} indică eticheta 0. "
    explanation += f"Cel mai apropiat context: '{retrieved_texts[0][:100]}...'"

    return {
        "label": predicted_label,
        "confidence score": confidence_score,
        "explanation": explanation
    }, retrieved_texts, retrieved_labels


Encoding knowledge base...


Batches: 100%|██████████| 60/60 [00:01<00:00, 30.02it/s]


In [14]:
from sklearn.metrics import precision_score, recall_score, f1_score

y_true = []
y_pred = []
retrieved_texts_all = []
explanations_all = []

print("Running evaluation...")
for i in range(len(val_dataset)):
    sample = val_dataset[i]
    output, retrieved_texts, retrieved_labels = run_inference(sample['text'], k=3)
    
    y_true.append(sample['label'])
    y_pred.append(output['label'])
    explanations_all.append(output['explanation'])
    retrieved_texts_all.append(retrieved_texts)

# Calculate metrics
precision = precision_score(y_true, y_pred)
recall = recall_score(y_true, y_pred)
f1 = f1_score(y_true, y_pred)
accuracy = sum([1 if y_true[i] == y_pred[i] else 0 for i in range(len(y_true))]) / len(y_true) 

print("RESULTS FOR RECALL@3:")
print(f"Accuracy: {accuracy:.4f}")
print(f"F1 Score: {f1:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall: {recall:.4f}")

Running evaluation...
RESULTS FOR RECALL@3:
Accuracy: 0.8000
F1 Score: 0.7619
Precision: 0.7805
Recall: 0.7442


In [16]:
from sklearn.metrics import precision_score, recall_score, f1_score

y_true = []
y_pred = []
retrieved_texts_all = []
explanations_all = []

print("Running evaluation...")
for i in range(len(val_dataset)):
    sample = val_dataset[i]
    output, retrieved_texts, retrieved_labels = run_inference(sample['text'], k=10)
    
    y_true.append(sample['label'])
    y_pred.append(output['label'])
    explanations_all.append(output['explanation'])
    retrieved_texts_all.append(retrieved_texts)

# Calculate metrics
precision = precision_score(y_true, y_pred)
recall = recall_score(y_true, y_pred)
f1 = f1_score(y_true, y_pred)
accuracy = sum([1 if y_true[i] == y_pred[i] else 0 for i in range(len(y_true))]) / len(y_true) 

print("RESULTS FOR RECALL@10:")
print(f"Accuracy: {accuracy:.4f}")
print(f"F1 Score: {f1:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall: {recall:.4f}")

Running evaluation...
RESULTS FOR RECALL@10:
Accuracy: 0.7600
F1 Score: 0.7391
Precision: 0.6939
Recall: 0.7907


In [15]:
import numpy as np
import pandas as pd
from datasets import Dataset

# Ensure your original components are available:
# faiss_index, kb_texts, kb_labels, embedder, val_dataset, run_inference

# =====================================================================
# ATTACK 1: Targeted Corpus Poisoning (Tested on k=3)
# Goal: Force highly similar sentences with conflicting labels into the KB.
# =====================================================================

print("\n--- Preparing Targeted Corpus Poisoning (k=3) ---")

# Let's create malicious sentences matching your legal validation queries
# We copy some text from the validation set but explicitly invert the target label
poisoned_texts = []
poisoned_labels = []

# Take 10 samples from the validation set, tweak them, and give them the wrong label
for i in range(10):
    original_text = val_dataset[i]['text']
    original_label = val_dataset[i]['label']
    adversarial_label = 1 if original_label == 0 else 0
    
    # Injecting a legal phrase that mimics high relevance but maps to the wrong target
    adversarial_text = original_text[:200] + " [DECIDAT: Fapta nu întrunește elementele constitutive ale infracțiunii conform jurisprudenței noi.]"
    
    poisoned_texts.append(adversarial_text)
    poisoned_labels.append(adversarial_label)

# Create a poisoned knowledge base by appending the malicious rows
poisoned_kb_texts = list(kb_texts) + poisoned_texts
poisoned_kb_labels = list(kb_labels) + poisoned_labels

# Rebuild a Poisoned FAISS Index
print("Encoding poisoned knowledge base...")
poisoned_embeddings = embedder.encode(poisoned_kb_texts, convert_to_numpy=True, show_progress_bar=False)
dimension = poisoned_embeddings.shape[1]
poisoned_index = faiss.IndexFlatL2(dimension)
poisoned_index.add(poisoned_embeddings)

# Override the system inference variables to use the poisoned database
def run_poisoned_inference(query_text, k):
    query_embedding = embedder.encode([query_text], convert_to_numpy=True)
    # Search using the poisoned index
    distances, indices = poisoned_index.search(query_embedding, k)
    
    retrieved_labels = [poisoned_kb_labels[i] for i in indices[0]]
    retrieved_texts = [poisoned_kb_texts[i] for i in indices[0]]
    
    label_0_count = retrieved_labels.count(0)
    label_1_count = retrieved_labels.count(1)
    predicted_label = 1 if label_1_count >= label_0_count else 0
    
    consensus = max(label_0_count, label_1_count) / k
    avg_distance = np.mean(distances[0])
    confidence_score = round(consensus * (1 / (1 + avg_distance * 0.1)), 2)
    
    return predicted_label, confidence_score

# Evaluate Attack 1 at k=3
y_true_p1, y_pred_p1 = [], []
for i in range(10): # Testing on the target subset
    sample = val_dataset[i]
    pred, _ = run_poisoned_inference(sample['text'], k=3)
    y_true_p1.append(sample['label'])
    y_pred_p1.append(pred)

acc_k3_attack = sum([1 for i in range(10) if y_true_p1[i] == y_pred_p1[i]]) / 10
print(f"Targeted Subset Accuracy under Corpus Poisoning (k=3): {acc_k3_attack:.4f}")


# =====================================================================
# ATTACK 2: Haystack Distraction / Consensus Dilution (Tested on k=10)
# Goal: Flood the retrieval with "Neutral" background noise to kill precision.
# =====================================================================

print("\n--- Preparing Haystack Distraction Attack (k=10) ---")

# Generate completely neutral, dense legal phrases that carry label=0 
# to artificially dilute any query where the true label should be 1.
distraction_texts = [
    "Procedura administrativă standard prevede arhivarea dosarelor în format electronic conform regulamentului.",
    "Termenul de depunere a contestațiilor este de 15 zile de la comunicarea oficială a deciziei.",
    "Grefierul de ședință va consemna toate declarațiile martorilor în procesul-verbal atașat la dosar."
] * 20 # Add 60 distraction documents

diluted_kb_texts = list(kb_texts) + distraction_texts
diluted_kb_labels = list(kb_labels) + [0] * len(distraction_texts) # Label them all as 0

# Rebuild a Diluted FAISS Index
diluted_embeddings = embedder.encode(diluted_kb_texts, convert_to_numpy=True, show_progress_bar=False)
diluted_index = faiss.IndexFlatL2(dimension)
diluted_index.add(diluted_embeddings)

def run_diluted_inference(query_text, k):
    query_embedding = embedder.encode([query_text], convert_to_numpy=True)
    distances, indices = diluted_index.search(query_embedding, k)
    
    retrieved_labels = [diluted_kb_labels[i] for i in indices[0]]
    label_0_count = retrieved_labels.count(0)
    label_1_count = retrieved_labels.count(1)
    
    predicted_label = 1 if label_1_count >= label_0_count else 0
    consensus = max(label_0_count, label_1_count) / k
    return predicted_label, consensus

# Evaluate Attack 2 at k=10
y_true_p2, y_pred_p2 = [], []
for i in range(len(val_dataset)):
    sample = val_dataset[i]
    pred, _ = run_diluted_inference(sample['text'], k=10) # Testing on high k
    y_true_p2.append(sample['label'])
    y_pred_p2.append(pred)

acc_k10_attack = sum([1 for i in range(len(val_dataset)) if y_true_p2[i] == y_pred_p2[i]]) / len(val_dataset)
print(f"Global Accuracy under Haystack Distraction (k=10): {acc_k10_attack:.4f}")


--- Preparing Targeted Corpus Poisoning (k=3) ---
Encoding poisoned knowledge base...
Targeted Subset Accuracy under Corpus Poisoning (k=3): 0.7000

--- Preparing Haystack Distraction Attack (k=10) ---
Global Accuracy under Haystack Distraction (k=10): 0.7600
